<a href="https://colab.research.google.com/github/Doran1690/llm-engineering-projects/blob/main/synthetic_data_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [6]:
MODEL = "meta-llama/Llama-3.2-3B-Instruct"

In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", quantization_config=quant_config)

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [11]:
messages = [
    {"role":"system","content":"You are a helpful assistant who generates realistic synthetic data. The data should look real and varied. Reply only with the data itself, without any explanations."},
    {"role":"user","content":"  Give me 5 customer reviews about a restaurant named Sicily Pizza."}
]

In [15]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=500)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

**5/5**
"I've been to Sicily Pizza 5 times this month and it never disappoints! The service is always friendly and the pizza is always hot and fresh. I highly recommend the Sicilian classic with mushrooms and olives!" - Rachel T.

**3/5**
"I stopped by Sicily Pizza for lunch on a whim and was pleasantly surprised. The atmosphere was cozy and the staff was attentive. However, my pizza was a bit overcooked and the toppings were a bit sparse. Still, I'd give it 3 stars for the value." - Mark K.

**5/5**
"Sicily Pizza is my go-to spot for pizza nights! The dough is always perfectly crispy and the sauce is made in-house. I love the unique topping combinations they offer - the fig and prosciutto pizza is a game-changer!" - Emily W.

**2/5**
"I was really looking forward to trying Sicily Pizza, but unfortunately, my experience was a letdown. The wait was long and the service was slow. The pizza was just okay, not bad, but not great either. Maybe I just caught them on a bad night?" - James B.


In [16]:
def generate_data(request):
    messages = [
        {"role": "system", "content": "You are a helpful assistant who generates realistic synthetic data. The data should look real and varied. Reply only with the data itself, without any explanations."},
        {"role": "user", "content": request}
    ]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=500)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

In [17]:
print(generate_data("Give me 5 fake customers with name, age and city."))

1. Emily Patel, 32, San Francisco
2. Landon Brooks, 41, Austin
3. Ava Moreno, 28, New York
4. Ethan Kim, 39, Chicago
5. Sophia Lee, 25, Los Angeles


In [18]:
import gradio as gr

gr.Interface(fn=generate_data, inputs="text", outputs="text").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://320b52dcdb60e2833b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
